# Lab 47 (solution): Trustworthy gold

Reference implementation. [Lab 45](../../45-anchoring-the-consensus/) anchored the annotator consensus to a gold label - but treated gold as a single expert's call. A single expert is just another annotator with its own bias. This builds gold from **multiple experts** (agreement measured, disagreements adjudicated), shows the annotator consensus was flattering the judge, and **re-derives the [Lab 40](../../40-annotation-quality/) judge ceiling against gold** instead of consensus.

Ships `expert_gold.jsonl` (three annotators + three experts + the judge label, per item).

## Step 0: Setup

In [ ]:
import json
from sklearn.metrics import cohen_kappa_score, accuracy_score

def fleiss_kappa(rows):
    """Fleiss kappa over per-item label counts [[n0,n1],...]; no statsmodels (Lab 40 style)."""
    N = len(rows)
    n = sum(rows[0])
    p=[sum(c[j] for c in rows)/(N*n) for j in range(len(rows[0]))]
    Pbar=sum((sum(ci*ci for ci in row)-n)/(n*(n-1)) for row in rows)/N
    Pe=sum(pj*pj for pj in p)
    return (Pbar-Pe)/(1-Pe) if Pe!=1 else 1.0
print("re-deriving the judge ceiling against multi-expert gold")

## Step 1: Annotators, experts, and the judge

In [ ]:
# Lab 45 anchored the consensus to a single gold label. But a single expert is just another
# annotator with its own bias. Here each item has three annotators (a1-a3), three EXPERTS
# (e1-e3), and the system's judge label.
with open("./expert_gold.jsonl") as f:
    items = [json.loads(line) for line in f]
def col(k):
    return [it[k] for it in items]
def maj(*cs):
    return [1 if sum(v) >= 2 else 0 for v in zip(*cs, strict=False)]
def counts(*cs):
    return [[len(cs) - sum(t), sum(t)] for t in zip(*cs, strict=False)]
consensus   = maj(col("a1"),col("a2"),col("a3"))     # annotator majority (Lab 45)
expert_gold = maj(col("e1"),col("e2"),col("e3"))     # adjudicated gold (majority of experts)
print(f"{len(items)} items, 3 annotators + 3 experts + judge")

## Step 2: Are experts a tighter anchor?

Inter-expert vs inter-annotator agreement.

In [ ]:
# Are experts a tighter anchor than annotators? Compare inter-rater agreement.
ann = fleiss_kappa(counts(col("a1"),col("a2"),col("a3")))
exp = fleiss_kappa(counts(col("e1"),col("e2"),col("e3")))
print(f"inter-annotator Fleiss kappa: {ann:.2f}")
print(f"inter-expert    Fleiss kappa: {exp:.2f}   <- experts agree more, so gold is a tighter anchor")
print("Multi-expert gold isn't perfect either - but its own ceiling (inter-expert agreement)")
print("is higher than the annotators', which is exactly why it can anchor them.")

## Step 3: A single expert is fallible

Why gold needs multiple experts + adjudication.

In [ ]:
# A single expert is fallible - that's why gold needs more than one, with adjudication.
for e in ["e1","e2","e3"]:
    print(f"single expert {e} vs adjudicated gold: {accuracy_score(expert_gold, col(e)):.2f}")
# adjudication queue: items where the experts split (need a senior decision / discussion)
split=[items[i]["id"] for i in range(len(items)) if len({items[i]["e1"],items[i]["e2"],items[i]["e3"]})>1]
print(f"expert adjudication queue (experts split): {split}")

## Step 4: Re-derive the judge ceiling against gold

In [ ]:
# Re-derive the Lab 40 judge ceiling. Lab 40 measured the judge against the annotator
# CONSENSUS. But if the judge shares the annotators' biases, that flatters it.
jc=cohen_kappa_score(col("judge"), consensus)
jg=cohen_kappa_score(col("judge"), expert_gold)
print(f"judge vs annotator consensus (Lab 40 ceiling proxy): {jc:.2f}")
print(f"judge vs expert gold        (re-derived ceiling):   {jg:.2f}")
print("\nThe judge looks strong against the consensus it correlates with, and weaker against")
print("gold. The consensus was grading the judge against its own mistakes. And expert gold")
print("recovers truth where the consensus failed:")
gold_true=[i%2 for i in range(len(items))]   # the constructed ground truth, for illustration
print(f"  consensus vs truth: {accuracy_score(gold_true,consensus):.2f}   gold vs truth: {accuracy_score(gold_true,expert_gold):.2f}")

## Step 5: The same lesson as the operations side

In [ ]:
# Same lesson as the operations side (Lab 46): a single point you trusted breaks when you
# look harder. One worker's state -> a shared store; one curated reference -> a sampled one;
# one corpus hash -> a per-doc map; and here, one consensus (or one expert) -> multi-expert,
# adjudicated gold. Anchor to something plural and external, then re-measure everything that
# was calibrated against the single point - including the judge ceiling.
print("Re-derive the ceiling against gold, not consensus. Re-grade the judge against gold.")
print("A ceiling measured against a flawed anchor is itself flawed.")

## Step 6: The discipline

In [ ]:
# The discipline: gold is not 'the expert said so'. Gold is multiple experts, their
# agreement measured, their disagreements adjudicated - and then every downstream metric
# (the judge ceiling, the annotator weights from Lab 45) re-derived against it.
print("Trustworthy gold is plural and adjudicated. Anything you calibrated against a single")
print("rater - human or model - is due for a recount.")

## What you built

Multi-expert gold and a re-derived judge ceiling: inter-expert agreement (Fleiss κ) measured against inter-annotator agreement to show experts are a tighter anchor; single-expert-vs-adjudicated-gold accuracy to show even experts err (so gold needs more than one, with an adjudication queue for splits); and the judge ceiling computed against expert gold rather than the annotator consensus - revealing that the consensus, which shares the judge's biases, was overstating the judge's quality.

**Where this simplifies:** three experts and 20 items is a teaching size (more of both for a real anchor); the labels are binary (real rubrics have graded or multi-class judgments); adjudication here is majority-of-experts (a real protocol routes splits to a senior adjudicator with the guideline in hand); and gold still isn't truth - it's a tighter, measurable anchor whose own ceiling (inter-expert agreement) you should report.

This closes the evaluation-quality thread end to end: [Lab 40](../../40-annotation-quality/) set a ceiling, [Lab 43](../../43-annotator-drift/) caught annotators drifting, [Lab 45](../../45-anchoring-the-consensus/) anchored the consensus to gold, and this makes the gold itself plural and adjudicated - then re-derives the ceiling against it.